# Instituto Tecnológico y de Estudios Superiores de Monterrey

## Análisis de Grandes Volúmenes de Datos

### Actividad 2 – Proyecto: Base de Datos de Big Data

---

### Equipo 45

**Integrantes:**

- Hiram García Austria  
- Edgar Oviedo Navarro  
- Fernando Pardo López  
- Alberto Cortés Murillo  

**Profesor:** Dr. Iván Olmos Pineda  
**Fecha de entrega:** 12 de mayo de 2026

---

## Objetivo de la actividad

El objetivo de esta actividad es caracterizar la población objetivo del dataset seleccionado, diseñar una estrategia de particionamiento basada en variables relevantes del dominio financiero e implementar una técnica de muestreo metodológicamente adecuada mediante PySpark, con el fin de construir subconjuntos representativos para etapas posteriores del proyecto.

---

## Contexto del proyecto

Como parte del proyecto del curso, el equipo seleccionó un dataset histórico del mercado bursátil que contiene información de más de 9,000 acciones con registros diarios desde 1962.

Este conjunto de datos incluye variables financieras clave como precios de apertura, cierre, máximos, mínimos, volumen de transacciones, dividendos y ajustes por división de acciones, representando más de 34 millones de registros y un tamaño aproximado de 4.47 GB.

Debido a su escala y complejidad, este dataset constituye un caso adecuado para la aplicación de herramientas de procesamiento distribuido como PySpark.

## 1. Caracterización de la población

La población objetivo corresponde al universo completo de registros históricos contenidos en el dataset bursátil seleccionado.

La unidad de análisis es cada registro diario asociado a un instrumento bursátil identificado mediante su ticker.

Variables principales consideradas:

- Date
- Ticker
- Open
- High
- Low
- Close
- Volume
- Dividends
- Stock Splits

Estas variables permiten describir comportamiento temporal, liquidez y dinámica financiera del mercado.

In [ ]:
import os
import findspark
import kagglehub

findspark.init()
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import expr, col, when, lit

spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better spark

c:\Users\hille\anaconda3\envs\env_pyspark\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Download latest version
path = kagglehub.dataset_download("jakewright/9000-tickers-of-stock-market-data-full-history")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\hille\.cache\kagglehub\datasets\jakewright\9000-tickers-of-stock-market-data-full-history\versions\2


In [6]:
path = path + "/all_stock_data.csv"
print(path)

C:\Users\hille\.cache\kagglehub\datasets\jakewright\9000-tickers-of-stock-market-data-full-history\versions\2/all_stock_data.csv


In [7]:
from pyspark.sql.types import *

# Definimos el esquema del data set
schema = StructType([

    StructField("Date", DateType(), True),
    StructField("Ticker", StringType(), True),

    StructField("Open", DoubleType(), True),
    StructField("High", DoubleType(), True),
    StructField("Low", DoubleType(), True),
    StructField("Close", DoubleType(), True),

    StructField("Volume", DoubleType(), True),
    StructField("Dividends", DecimalType(5,1), True),
    StructField("Stock Splits", DecimalType(5,1), True)

])

In [8]:
# Disparador 0
# Leemos el data set e imprimimos los primeros 5 registros

df = spark.read.csv(path, header=True, inferSchema=False, schema=schema)
df.show(5)

+----------+------+----+-------------------+-------------------+-------------------+---------+---------+------------+
|      Date|Ticker|Open|               High|                Low|              Close|   Volume|Dividends|Stock Splits|
+----------+------+----+-------------------+-------------------+-------------------+---------+---------+------------+
|1962-01-02|    ED| 0.0| 0.2658275556233194|0.26178762316703796|0.26178762316703796|  25600.0|      0.0|         0.0|
|1962-01-02|   CVX| 0.0|0.04680890217423439|0.04606926600933256|0.04680890217423439| 105840.0|      0.0|         0.0|
|1962-01-02|    GD| 0.0|0.21003275954390174|0.20306070787008793| 0.2082897424697876|2648000.0|      0.0|         0.0|
|1962-01-02|    BP| 0.0|0.14143933090345925|0.13952797651290894|0.13952797651290894|  77440.0|      0.0|         0.0|
|1962-01-02|   MSI| 0.0| 0.7649229763450202| 0.7452535214492476| 0.7518101930618286|  65671.0|      0.0|         0.0|
+----------+------+----+-------------------+------------

In [9]:
df.printSchema()

root
 |-- Date: date (nullable = true)
 |-- Ticker: string (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: double (nullable = true)
 |-- Dividends: decimal(5,1) (nullable = true)
 |-- Stock Splits: decimal(5,1) (nullable = true)



In [10]:
from pyspark.sql.functions import count, when, col

# Definimos el cache
df.cache()

resumen = df.describe()

valores_nulos = df.select([
    count(when(col(c).isNull(), 1)).alias(c)
    for c in df.columns
])



In [11]:
# Disparador 1
num = df.count()

In [12]:
# Imprimimos el numero de columnas y registros

print(f"Número de columnas: {len(df.columns)}")
print(f"Número de registros: {num:,}\n")


Número de columnas: 9
Número de registros: 34,646,258



In [13]:
# Disparador 2
resumen_transpuesto = resumen.toPandas().set_index('summary').T

In [14]:
# Imprimimos el  resumen
resumen_transpuesto

summary,count,mean,stddev,min,max
Ticker,34646258,NaN,NaN,A,ZZLL
Open,34646149,1.1150488437326563E23,3.954734941003448E25,-8.210044423351318E25,1.5072897744279783E28
High,34646149,1.1150488437326608E23,3.954734941003448E25,-8.210044423351318E25,1.5072897744279783E28
Low,34646149,1.115048843732678E23,3.954734941003448E25,-8.210044423351318E25,1.5072897744279783E28
Close,34646152,1.1150487471809343E23,3.95473476978514E25,-8.210044423351318E25,1.5072897744279783E28
Volume,34646258,1339229.8683998429,1.5671698842802074E7,0.0,9.230856E9
Dividends,34646258,0.00401,1.6035252555952273,0.0,4500.0
Stock Splits,34646255,0.00039,0.1896345452067494,0.0,1000.0


In [15]:
# Disparador 3
nulos_pandas = valores_nulos.toPandas().T

In [16]:
# Imprimimos los valores nulos
nulos_pandas.columns = ['Valores nulos']
nulos_pandas["%"] = (nulos_pandas["Valores nulos"] / num) * 100

nulos_pandas


,Valores nulos,%
Date,0,0.000000
Ticker,0,0.000000
Open,109,0.000315
High,109,0.000315
Low,109,0.000315
Close,106,0.000306
Volume,0,0.000000
Dividends,0,0.000000
Stock Splits,3,0.000009


```markdown
# Estrategia de Particionamiento

Vamos a implementar la estrategia de particionamiento descrita, utilizando las variables **Periodo económico**, **Nivel de volatilidad**, y **Volumen de transacciones**.

## 1. Variable: Periodo Económico
Primero, definiremos una función para clasificar las fechas en 'Pre-crisis', 'Crisis' o 'Recuperación' basándonos en los eventos macroeconómicos mencionados.
```

# 2. Estrategia de Particionamiento

Con base en la caracterización previa del dataset, se implementa una estrategia de particionamiento utilizando tres variables representativas del comportamiento del mercado: **periodo económico**, **nivel de volatilidad** y **volumen de transacciones**.

El objetivo es segmentar la población en subconjuntos homogéneos que permitan aplicar posteriormente técnicas de muestreo con mayor consistencia estadística y representatividad.

## 2.1. Variable: Periodo Económico

Como primera dimensión de segmentación, se clasificarán los registros según periodos económicos relevantes, considerando eventos macroeconómicos que impactaron el comportamiento general del mercado financiero.

In [17]:
from pyspark.sql.functions import when, col, to_date, lit

# Definición de periodos económicos relevantes para segmentación del dataset
crisis_periods = [
    ("1987-10-01", "1988-03-31", "Crisis"), # Lunes negro
    ("2000-03-01", "2001-11-30", "Crisis"), # Burbuja de las punto com
    ("2008-09-01", "2009-03-31", "Crisis"), # Crisis financiera
    ("2020-02-01", "2020-05-31", "Crisis"), # Pandemia del Covid
    ("2022-02-01", "2023-01-31", "Crisis")  # Guerra Rusia vs Ucrania (fin arbitrario para el ejemplo)
]

# Definir periodos de recuperación (aproximados)
recovery_periods = [
    ("1988-04-01", "2000-02-29", "Recuperación"), # Después de Lunes negro, antes de Dot-com
    ("2001-12-01", "2008-08-31", "Recuperación"), # Después de Dot-com, antes de Crisis financiera
    ("2009-04-01", "2020-01-31", "Recuperación"), # Después de Crisis financiera, antes de Covid
    ("2020-06-01", "2022-01-31", "Recuperación"), # Después de Covid, antes de Guerra
    ("2023-02-01", "2024-12-31", "Recuperación")  # Después de Guerra (arbitrario para el futuro)
]

# Inicializar la columna 'Periodo_Economico' con 'Pre-crisis' como valor por defecto
df_partitioned = df.withColumn("Periodo_Economico", lit("Pre-crisis"))

# Aplicar las condiciones para Periodos de Crisis
for start_date_str, end_date_str, period_type in crisis_periods:
    start_date = to_date(lit(start_date_str))
    end_date = to_date(lit(end_date_str))
    df_partitioned = df_partitioned.withColumn("Periodo_Economico",
                                                when((col("Date") >= start_date) & (col("Date") <= end_date), lit(period_type))
                                                .otherwise(col("Periodo_Economico")))

# Aplicar las condiciones para Periodos de Recuperación
for start_date_str, end_date_str, period_type in recovery_periods:
    start_date = to_date(lit(start_date_str))
    end_date = to_date(lit(end_date_str))
    df_partitioned = df_partitioned.withColumn("Periodo_Economico",
                                                when((col("Date") >= start_date) & (col("Date") <= end_date), lit(period_type))
                                                .otherwise(col("Periodo_Economico")))

# Mostrar la distribución de los periodos económicos
df_partitioned.groupBy("Periodo_Economico").count().show()

+-----------------+--------+
|Periodo_Economico|   count|
+-----------------+--------+
|       Pre-crisis| 1560048|
|           Crisis| 4206011|
|     Recuperación|28880199|
+-----------------+--------+



## 2.2 Variable: Nivel de Volatilidad

Se calcula la volatilidad diaria como la variación porcentual entre los precios máximo (*High*) y mínimo (*Low*), clasificando posteriormente los registros en niveles de volatilidad: *Baja*, *Media* y *Alta*.

In [ ]:
# Mostrar la distribución de los niveles de volatilidad
df_partitioned = df_partitioned.withColumn(
    "Volatilidad_Diaria_Pct",
    expr("try_divide((High - Low) * 100, Low)")
)

df_partitioned = df_partitioned.withColumn(
    "Nivel_Volatilidad",
    when(col("Volatilidad_Diaria_Pct").isNull(), lit("Sin dato"))
    .when(col("Volatilidad_Diaria_Pct") < 2, lit("Baja"))
    .when((col("Volatilidad_Diaria_Pct") >= 2) & (col("Volatilidad_Diaria_Pct") <= 5), lit("Media"))
    .otherwise(lit("Alta"))
)

df_partitioned.groupBy("Nivel_Volatilidad").count().show()

+-----------------+--------+
|Nivel_Volatilidad|   count|
+-----------------+--------+
|             Alta| 7621183|
|            Media|10781440|
|         Sin dato|    1059|
|             Baja|16242576|
+-----------------+--------+



## 2.3. Variable: Volumen de Transacciones

Se calculan percentiles sobre la variable de volumen de transacciones para clasificar los registros en niveles relativos de volumen de operación: Bajo, Medio y Alto.

In [23]:
# Calcular los percentiles para la columna 'Volume'
# Usamos approx_percentile para DataFrames grandes
volume_percentiles = df_partitioned.approxQuantile("Volume", [0.33, 0.66], 0.01)
p33 = volume_percentiles[0]
p66 = volume_percentiles[1]

print(f"Percentil 33 de Volumen: {p33:,.2f}")
print(f"Percentil 66 de Volumen: {p66:,.2f}")

df_partitioned = df_partitioned.withColumn("Volumen_Transacciones",
                                           when(col("Volume") <= p33, lit("Bajo"))
                                           .when((col("Volume") > p33) & (col("Volume") <= p66), lit("Medio"))
                                           .otherwise(lit("Alto")))

# Mostrar la distribución de los volúmenes de transacciones
df_partitioned.groupBy("Volumen_Transacciones").count().show()

Percentil 33 de Volumen: 9,200.00
Percentil 66 de Volumen: 200,100.00
+---------------------+--------+
|Volumen_Transacciones|   count|
+---------------------+--------+
|                Medio|11418009|
|                 Alto|11869581|
|                 Bajo|11358668|
+---------------------+--------+



## 3. Combinaciones de estratos generadas

A partir de las tres variables de segmentación seleccionadas (*Periodo Económico*, *Nivel de Volatilidad* y *Volumen de Transacciones*), se generan combinaciones de estratos para representar subconjuntos homogéneos del dataset.

Dado que cada variable posee tres categorías, el número máximo teórico de combinaciones posibles es:

3 × 3 × 3 = 27 estratos.

Sin embargo, el número real dependerá de las combinaciones efectivamente presentes en los datos históricos analizados.

In [24]:
from pyspark.sql.functions import count, lit, round

total_registros = df_partitioned.count()

df_estratos = df_partitioned.groupBy(
    "Periodo_Economico",
    "Nivel_Volatilidad",
    "Volumen_Transacciones"
).count()

df_estratos = df_estratos.withColumn(
    "Porcentaje",
    round((col("count") / lit(total_registros)) * 100, 4)
)

df_estratos.orderBy(col("Porcentaje").desc()).show(30, truncate=False)

+-----------------+-----------------+---------------------+-------+----------+
|Periodo_Economico|Nivel_Volatilidad|Volumen_Transacciones|count  |Porcentaje|
+-----------------+-----------------+---------------------+-------+----------+
|Recuperación     |Baja             |Bajo                 |6621937|19.113    |
|Recuperación     |Media            |Alto                 |4390511|12.6724   |
|Recuperación     |Baja             |Medio                |3841333|11.0873   |
|Recuperación     |Baja             |Alto                 |3473234|10.0248   |
|Recuperación     |Media            |Medio                |3354626|9.6825    |
|Recuperación     |Alta             |Medio                |2256682|6.5135    |
|Recuperación     |Alta             |Alto                 |2025171|5.8453    |
|Recuperación     |Alta             |Bajo                 |1656268|4.7805    |
|Recuperación     |Media            |Bajo                 |1259405|3.635     |
|Crisis           |Baja             |Bajo           

## 4. Combinación de Particiones
Ahora, podemos ver cómo se combinan estas variables para formar las particiones. Mostraremos los primeros registros del DataFrame con las nuevas columnas de particionamiento.

In [25]:
df_partitioned.select("Date", "Ticker", "Volumen_Transacciones", "Nivel_Volatilidad", "Periodo_Economico").show(10)

# Opcionalmente, puedes contar las ocurrencias de una combinación de particiones específica
df_partitioned.groupBy("Periodo_Economico", "Nivel_Volatilidad", "Volumen_Transacciones").count().show(10, truncate=False)

+----------+------+---------------------+-----------------+-----------------+
|      Date|Ticker|Volumen_Transacciones|Nivel_Volatilidad|Periodo_Economico|
+----------+------+---------------------+-----------------+-----------------+
|1962-01-02|    ED|                Medio|             Baja|       Pre-crisis|
|1962-01-02|   CVX|                Medio|             Baja|       Pre-crisis|
|1962-01-02|    GD|                 Alto|            Media|       Pre-crisis|
|1962-01-02|    BP|                Medio|             Baja|       Pre-crisis|
|1962-01-02|   MSI|                Medio|            Media|       Pre-crisis|
|1962-01-02|   HON|                Medio|             Baja|       Pre-crisis|
|1962-01-02|    FL|                Medio|             Baja|       Pre-crisis|
|1962-01-02|    GT|                Medio|             Baja|       Pre-crisis|
|1962-01-02|   JNJ|                 Bajo|             Baja|       Pre-crisis|
|1962-01-02|   MMM|                 Alto|            Media|     

## 5. Distribución porcentual de los estratos

Se calcula la proporción de ocurrencia de cada combinación generada para utilizarla como base del muestreo estratificado proporcional.

In [26]:
from pyspark.sql.functions import count, lit, round

total_registros = df_partitioned.count()

df_estratos = df_partitioned.groupBy(
    "Periodo_Economico",
    "Nivel_Volatilidad",
    "Volumen_Transacciones"
).count()

df_estratos = df_estratos.withColumn(
    "Porcentaje",
    round((col("count") / lit(total_registros)) * 100, 4)
)

df_estratos.orderBy(col("Porcentaje").desc()).show(27, truncate=False)

+-----------------+-----------------+---------------------+-------+----------+
|Periodo_Economico|Nivel_Volatilidad|Volumen_Transacciones|count  |Porcentaje|
+-----------------+-----------------+---------------------+-------+----------+
|Recuperación     |Baja             |Bajo                 |6621937|19.113    |
|Recuperación     |Media            |Alto                 |4390511|12.6724   |
|Recuperación     |Baja             |Medio                |3841333|11.0873   |
|Recuperación     |Baja             |Alto                 |3473234|10.0248   |
|Recuperación     |Media            |Medio                |3354626|9.6825    |
|Recuperación     |Alta             |Medio                |2256682|6.5135    |
|Recuperación     |Alta             |Alto                 |2025171|5.8453    |
|Recuperación     |Alta             |Bajo                 |1656268|4.7805    |
|Recuperación     |Media            |Bajo                 |1259405|3.635     |
|Crisis           |Baja             |Bajo           

## 6. Técnica de muestreo por partición

Una vez construidas las particiones mediante las variables periodo económico, nivel de volatilidad y volumen de transacciones, se propone aplicar una técnica de muestreo estratificado.

Cada combinación de estas variables representa un estrato de la población. Por ello, el muestreo estratificado permite seleccionar registros de cada partición, conservando la diversidad de escenarios presentes en el dataset.

Esta técnica es adecuada porque evita que la muestra quede dominada por los grupos con mayor número de registros y ayuda a preservar información de periodos menos frecuentes, como crisis o escenarios de alta volatilidad.

In [27]:
from pyspark.sql.functions import concat_ws

df_partitioned = df_partitioned.withColumn(
    "Estrato",
    concat_ws(
        "_",
        col("Periodo_Economico"),
        col("Nivel_Volatilidad"),
        col("Volumen_Transacciones")
    )
)

df_partitioned.groupBy("Estrato").count().orderBy("count", ascending=False).show(30, truncate=False)

+---------------------------+-------+
|Estrato                    |count  |
+---------------------------+-------+
|Recuperación_Baja_Bajo     |6621937|
|Recuperación_Media_Alto    |4390511|
|Recuperación_Baja_Medio    |3841333|
|Recuperación_Baja_Alto     |3473234|
|Recuperación_Media_Medio   |3354626|
|Recuperación_Alta_Medio    |2256682|
|Recuperación_Alta_Alto     |2025171|
|Recuperación_Alta_Bajo     |1656268|
|Recuperación_Media_Bajo    |1259405|
|Crisis_Baja_Bajo           |893429 |
|Crisis_Alta_Alto           |658391 |
|Crisis_Media_Alto          |631243 |
|Crisis_Alta_Medio          |542005 |
|Crisis_Media_Medio         |457087 |
|Crisis_Baja_Medio          |357435 |
|Pre-crisis_Baja_Medio      |316259 |
|Crisis_Alta_Bajo           |295893 |
|Pre-crisis_Baja_Bajo       |286782 |
|Pre-crisis_Baja_Alto       |245539 |
|Pre-crisis_Media_Medio     |223470 |
|Crisis_Baja_Alto           |206628 |
|Pre-crisis_Media_Alto      |201031 |
|Crisis_Media_Bajo          |163874 |
|Pre-crisis_

In [28]:
estratos = [row["Estrato"] for row in df_partitioned.select("Estrato").distinct().collect()]

fractions = {estrato: 0.05 for estrato in estratos}

sampled_df = df_partitioned.sampleBy(
    "Estrato",
    fractions=fractions,
    seed=42
)

print("Total población particionada:", df_partitioned.count())
print("Total muestra obtenida:", sampled_df.count())

sampled_df.groupBy("Estrato").count().orderBy("count", ascending=False).show(30, truncate=False)

Total población particionada: 34646258
Total muestra obtenida: 1732359
+---------------------------+------+
|Estrato                    |count |
+---------------------------+------+
|Recuperación_Baja_Bajo     |330629|
|Recuperación_Media_Alto    |219466|
|Recuperación_Baja_Medio    |191471|
|Recuperación_Baja_Alto     |174095|
|Recuperación_Media_Medio   |167555|
|Recuperación_Alta_Medio    |113287|
|Recuperación_Alta_Alto     |101285|
|Recuperación_Alta_Bajo     |82543 |
|Recuperación_Media_Bajo    |63015 |
|Crisis_Baja_Bajo           |44805 |
|Crisis_Alta_Alto           |33114 |
|Crisis_Media_Alto          |31635 |
|Crisis_Alta_Medio          |27391 |
|Crisis_Media_Medio         |22918 |
|Crisis_Baja_Medio          |17682 |
|Pre-crisis_Baja_Medio      |15934 |
|Crisis_Alta_Bajo           |14846 |
|Pre-crisis_Baja_Bajo       |14239 |
|Pre-crisis_Baja_Alto       |12240 |
|Pre-crisis_Media_Medio     |11113 |
|Pre-crisis_Media_Alto      |10181 |
|Crisis_Baja_Alto           |10151 |
|Cri